# PTM screen analysis

Post hoc summaries of the completed phospho/glyco screen: marginal and protein-conditioned associations, within-protein heterogeneity, diagnostic review, available sensitivity comparisons, signature context and precision. Uses saved aggregate tables; no model refitting. Configure the completed-run path in the first code cell.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, zipfile, hashlib, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

INPUT_ZIP = Path(globals().get("INPUT_ZIP", "/content/drive/MyDrive/new_dysfunction/multiomics_extensions/protein_ptm_paired_v2/publication/2687095c8b4e1e44/shards_4/aggregate/Protein_PTM_reports.zip"))
DOWNLOAD = globals().get("DOWNLOAD", True)
if str(INPUT_ZIP).startswith('/content/drive/'):
    from google.colab import drive
    drive.mount('/content/drive')
STAMP = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT = Path(globals().get('OUTPUT_DIR', INPUT_ZIP.parent / ('deeper_analysis_' + STAMP)))
OUT.mkdir(parents=True, exist_ok=False)
(OUT/'tables').mkdir(); (OUT/'plots').mkdir()
MANIFEST=[]
def export(name, df, purpose):
    df.to_csv(OUT/'tables'/f'{name}.csv', index=False)
    MANIFEST.append(dict(file=f'tables/{name}.csv', rows=len(df), purpose=purpose))
    return df

def boolean(s):
    text=s.astype(str).str.lower().str.strip()
    if not text.isin(['true','false','1','0']).all():
        raise ValueError(f'Invalid boolean values in {s.name}')
    return text.isin(['true','1'])

def direction(df):
    return np.where(df.hdi_95_lower>0, 'positive', np.where(df.hdi_95_upper<0, 'negative', 'uncertain'))

def mdtable(df):
    rows=[list(df.columns)]+df.fillna('').astype(str).values.tolist()
    lines=['| '+' | '.join(str(v).replace('|','/').replace(chr(10),' ') for v in row)+' |' for row in rows]
    lines.insert(1,'| '+' | '.join(['---']*len(df.columns))+' |')
    return chr(10).join(lines)

def plot_save(name):
    plt.tight_layout(); plt.savefig(OUT/'plots'/f'{name}.png', dpi=160, bbox_inches='tight')
    plt.show(); plt.close()


In [ ]:
required=['all_effects_95HDI.csv','eligible_features.csv','fit_status.csv','predictive_checks.csv','influence_status.csv','site_signature_mapping.csv','completion_summary.json','contract.json','planned_jobs.csv']
with zipfile.ZipFile(INPUT_ZIP) as z:
    missing=set(required)-set(z.namelist())
    if missing: raise FileNotFoundError(sorted(missing))
    A=pd.read_csv(z.open(required[0])); E=pd.read_csv(z.open(required[1])); F=pd.read_csv(z.open(required[2]))
    PC=pd.read_csv(z.open(required[3])); I=pd.read_csv(z.open(required[4])); S=pd.read_csv(z.open(required[5]))
    COMPLETION=json.loads(z.read('completion_summary.json')); CONTRACT=json.loads(z.read('contract.json'))
    JOBS=pd.read_csv(z.open('planned_jobs.csv'))
    D=pd.read_csv(z.open('patient_deletion_changes.csv')) if 'patient_deletion_changes.csv' in z.namelist() else pd.DataFrame()
    B=pd.read_csv(z.open('bootstrap_target_linear_benchmark.csv')) if 'bootstrap_target_linear_benchmark.csv' in z.namelist() else pd.DataFrame()
for df, cols in [(A,['numerical_checks_pass','hdi_excludes_zero']), (F,['numerical_checks_pass','postprocessing_complete']), (PC,['flag']), (I,['deletion_numerically_adequate'])]:
    for col in cols: df[col]=boolean(df[col])
assert not A.duplicated(['key','variant','quantity','term']).any()
assert not F.duplicated(['key','variant']).any() and E.key.is_unique
assert len(F)==COMPLETION['finished_fits']==len(JOBS)
assert set(zip(F.key,F.variant))==set(zip(JOBS.key,JOBS.variant))
assert F.postprocessing_complete.all()
assert np.isfinite(A[['posterior_mean','hdi_95_lower','hdi_95_upper']]).all().all()
assert (A.hdi_95_lower<=A.hdi_95_upper).all()
assert (A.hdi_excludes_zero == ((A.hdi_95_lower>0)|(A.hdi_95_upper<0))).all()
check=A.merge(F[['key','variant','numerical_checks_pass']],on=['key','variant'],suffixes=('_effect','_status'),validate='many_to_one')
assert (check.numerical_checks_pass_effect==check.numerical_checks_pass_status).all()
P=A[(A.variant=='primary') & (A.quantity=='PTM_given_Protein') & (A.term=='bmi_5_within')].copy()
assert P.key.is_unique and set(P.key)==set(E.key)
P['direction']=direction(P)
P['candidate']=P.numerical_checks_pass & P.direction.ne('uncertain')
qc=F[F.variant.eq('primary')][['key','max_rhat','min_ess_bulk','min_ess_tail','min_bfmi','divergences','maximum_depth_hits','predictive_flags','PSIS_flagged_patients']]
P=P.merge(qc,on='key',validate='one_to_one')
if len(I):
    iq=I.groupby('key').agg(deletion_pairs=('patient','size'),unresolved_deletions=('deletion_numerically_adequate',lambda s:int((~s).sum()))).reset_index()
else: iq=pd.DataFrame(columns=['key','deletion_pairs','unresolved_deletions'])
P=P.merge(iq,on='key',how='left',validate='one_to_one')
P[['deletion_pairs','unresolved_deletions']]=P[['deletion_pairs','unresolved_deletions']].fillna(0).astype(int)
assert (P.deletion_pairs==P.PSIS_flagged_patients).all(), 'Influence inventory mismatch'
P['needs_diagnostic_review']=(P.predictive_flags>0)|(P.unresolved_deletions>0)
P['hdi_width']=P.hdi_95_upper-P.hdi_95_lower
P['absolute_mean']=P.posterior_mean.abs()
P['distance_interval_from_zero']=np.where(P.direction.eq('positive'),P.hdi_95_lower,np.where(P.direction.eq('negative'),-P.hdi_95_upper,0))
export('all_primary_features',P,'All primary conditional estimates, including numerical failures and diagnostic flags')
Q=P[P.numerical_checks_pass].copy(); C=Q[Q.candidate].copy()
export('candidates_with_diagnostics',C,'Pointwise interval exclusion; no multiplicity control or reliability ranking')
summary=P.groupby('assay').agg(eligible=('key','size'),genes=('gene','nunique'),numerically_adequate=('numerical_checks_pass','sum'),candidate_features=('candidate','sum')).reset_index()
summary['candidate_genes']=summary.assay.map(C.groupby('assay').gene.nunique()).fillna(0).astype(int)
summary['candidate_fraction_of_adequate']=summary.candidate_features/summary.numerically_adequate
export('scope_summary',summary,'Assay-specific counts and denominators'); display(summary)


In [ ]:
M=A[(A.variant=='primary') & (A.quantity=='PTM') & (A.term=='bmi_5_within')].copy()
M['marginal_direction']=direction(M)
T=Q.merge(M[['key','posterior_mean','hdi_95_lower','hdi_95_upper','marginal_direction']],on='key',validate='one_to_one',suffixes=('_conditional','_marginal'))
T['mean_change']=T.posterior_mean_conditional-T.posterior_mean_marginal
T['posterior_mean_sign_change']=(T.posterior_mean_conditional*T.posterior_mean_marginal)<0
T['absolute_mean_reduction']=T.posterior_mean_marginal.abs()-T.posterior_mean_conditional.abs()
T['interval_transition']=T.marginal_direction+' → '+T.direction
export('marginal_conditional_feature_comparison',T,'Matched primary estimates; descriptive differences only')
trans=T.groupby(['assay','marginal_direction','direction']).size().rename('features').reset_index()
export('interval_transition_counts',trans,'Counts of all 3×3 interval categories'); display(trans)
fig,axes=plt.subplots(1,2,figsize=(10,4))
for ax,(assay,g) in zip(axes,T.groupby('assay',sort=True)):
    ax.scatter(g.posterior_mean_marginal,g.posterior_mean_conditional,c=np.where(g.candidate,'#D81B60','#9DA3AA'),s=10,alpha=.6)
    lim=max(g.posterior_mean_marginal.abs().max(),g.posterior_mean_conditional.abs().max())*1.1
    ax.plot([-lim,lim],[-lim,lim],'k--',lw=.8); ax.axhline(0,color='grey',lw=.5);ax.axvline(0,color='grey',lw=.5)
    ax.set(title=assay,xlabel='Marginal PTM BMI slope',ylabel='Protein-conditioned BMI slope')
plot_save('marginal_conditional')


In [ ]:
G=Q.groupby(['assay','gene']).agg(features=('key','size'),positive_means=('posterior_mean',lambda s:int((s>0).sum())),negative_means=('posterior_mean',lambda s:int((s<0).sum())),positive_intervals=('direction',lambda s:int((s=='positive').sum())),negative_intervals=('direction',lambda s:int((s=='negative').sum())),uncertain_intervals=('direction',lambda s:int((s=='uncertain').sum())),candidate_features=('candidate','sum'),predictive_flagged_features=('predictive_flags',lambda s:int((s>0).sum())),unresolved_features=('unresolved_deletions',lambda s:int((s>0).sum())),median_feature_mean=('posterior_mean','median'),min_feature_mean=('posterior_mean','min'),max_feature_mean=('posterior_mean','max')).reset_index()
G['candidate_fraction']=G.candidate_features/G.features
G['majority_mean_fraction']=G[['positive_means','negative_means']].max(axis=1)/G.features
G['opposing_candidate_directions']=(G.positive_intervals>0)&(G.negative_intervals>0)
export('all_gene_descriptive_summary',G,'Feature-level descriptive summaries; median is not a fitted protein effect')
export('candidate_gene_full_feature_context',Q.merge(G.loc[G.candidate_features>0,['assay','gene']],on=['assay','gene'],validate='many_to_one'),'All adequate features of candidate genes, including uncertain features')
display(G[G.candidate_features>0].sort_values(['assay','gene']))
# Characterise feature identifiers without assigning unverified glycosites or sugar meanings.
ID=E[['key','assay','gene','feature_id']].copy()
ID['glycan_composition']=ID.feature_id.str.extract(r'-(N[0-9]+H[0-9]+F[0-9]+S[0-9]+G[0-9]+)$',expand=False)
ID['peptide_source_string']=np.where(ID.assay.eq('glyco'),ID.feature_id.str.replace(r'-N[0-9]+H[0-9]+F[0-9]+S[0-9]+G[0-9]+$','',regex=True),None)
export('feature_identity_map',ID,'Original IDs retained; composition labels are not structural or localization assignments')
GP=Q.merge(ID[['key','peptide_source_string','glycan_composition']],on='key',validate='one_to_one')
GP=GP[GP.assay.eq('glyco')]
pg=GP.groupby(['gene','peptide_source_string'],dropna=False).agg(features=('key','size'),positive_intervals=('direction',lambda s:int((s=='positive').sum())),negative_intervals=('direction',lambda s:int((s=='negative').sum())),positive_means=('posterior_mean',lambda s:int((s>0).sum())),negative_means=('posterior_mean',lambda s:int((s<0).sum()))).reset_index()
pg['opposing_candidate_directions']=(pg.positive_intervals>0)&(pg.negative_intervals>0)
export('same_peptide_glycoform_context',pg,'Descriptive heterogeneity within exact source peptide strings; no interconversion or composition-ratio test')


In [ ]:
export('candidate_diagnostic_review',C[C.needs_diagnostic_review],'Candidates requiring predictive/influence review')
export('candidates_without_recorded_flags',C[~C.needs_diagnostic_review],'Descriptive restriction; absence of flags does not prove model correctness')
PS=PC[PC.variant.eq('primary')].merge(E[['key','assay']],on='key',validate='many_to_one')
ps=PS.groupby(['assay','check']).agg(checks=('key','size'),flagged=('flag','sum'),median_observed=('observed_draw_median','median')).reset_index()
export('primary_predictive_summary',ps,'Predictive checks kept separate from numerical convergence')
if len(D):
    DD=D[(D.quantity=='PTM_given_Protein') & (D.term=='bmi_5_within')].copy()
    DD['direction_primary']=np.where(DD.hdi_95_lower_primary>0,'positive',np.where(DD.hdi_95_upper_primary<0,'negative','uncertain'))
    DD['direction_deletion']=np.where(DD.hdi_95_lower_deletion>0,'positive',np.where(DD.hdi_95_upper_deletion<0,'negative','uncertain'))
    DD['interval_class_changed']=DD.direction_primary!=DD.direction_deletion
    DD['mean_sign_changed']=DD.posterior_mean_primary*DD.posterior_mean_deletion<0
    DD['absolute_mean_change']=abs(DD.posterior_mean_deletion-DD.posterior_mean_primary)
    DD['change_over_primary_hdi_width']=DD.absolute_mean_change/(DD.hdi_95_upper_primary-DD.hdi_95_lower_primary)
    export('all_available_deletion_comparisons',DD,'Adequate deletion refits only; no claim of complete leave-one-out analysis')
    CD=DD[DD.key.isin(C.key)].copy()
    export('candidate_deletion_comparisons',CD,'Candidate-specific effect changes for available adequate deletion refits')
    display(CD[['gene_primary','deleted_patient','absolute_mean_change','direction_primary','direction_deletion']])
else:
    DD=pd.DataFrame(); CD=pd.DataFrame()
export('unresolved_deletions',I[~I.deletion_numerically_adequate],'Unresolved checks preserved, not counted as stable')


In [ ]:
V=A[(A.quantity=='PTM_given_Protein') & (A.term=='bmi_5_within')].copy()
V['direction']=direction(V)
comparisons=[('primary','fixed_plex'),('primary','stage_subset_base'),('stage_subset_base','stage_adjusted'),('primary','wes_subset_base'),('wes_subset_base','wes_adjusted'),('primary','mutation_adjusted'),('primary','exclude_recorded_weight_loss'),('primary','exclude_adenosquamous')]
parts=[]
for base,variant in comparisons:
    x=V[V.variant.eq(base)].merge(V[V.variant.eq(variant)],on=['key','assay','gene','feature_id'],suffixes=('_base','_comparison'),validate='one_to_one')
    x['comparison']=base+' → '+variant
    x['mean_change']=x.posterior_mean_comparison-x.posterior_mean_base
    x['both_numerically_adequate']=x.numerical_checks_pass_base & x.numerical_checks_pass_comparison
    x['interval_class_changed']=x.direction_base!=x.direction_comparison
    parts.append(x)
VS=pd.concat(parts,ignore_index=True)
export('available_continuous_sensitivities',VS,'Descriptive mean/interval comparisons; no inferred contrast HDI')
CAT=A[(A.quantity=='PTM_given_Protein') & A.variant.eq('categorical')].copy()
export('categorical_target_contrasts',CAT,'Group SD differences; not continuous slopes or independent replication')
avail=V[V.variant.ne('primary')].groupby('key').variant.agg(lambda x:';'.join(sorted(set(x))))
gaps=C[['key','assay','gene','feature_id']].copy();gaps['available_variants']=gaps.key.map(avail).fillna('none')
export('candidate_sensitivity_availability',gaps,'Missing sensitivity fits are not evidence of robustness')
display(gaps.groupby(['assay','available_variants']).size().rename('features').reset_index())
if len(B):
    BB=B[B.quantity.eq('PTM_given_Protein')].merge(P[['key','posterior_mean','hdi_95_lower','hdi_95_upper','numerical_checks_pass']],on='key',validate='one_to_one')
    BB['point_estimate_difference']=BB.posterior_mean-BB.estimate
    export('target_bootstrap_comparison',BB,'Fixed-plex linear benchmark only; different estimator and interval type')


In [ ]:
links=S[['key','gene','family','CellType','Signature','global_feature']].drop_duplicates()
L=links.merge(Q[['key','candidate','direction']],on='key',validate='many_to_one')
rows=[]
for labels,g in L.groupby(['family','CellType','Signature','global_feature']):
    c=g[g.candidate]
    rows.append(dict(zip(['family','CellType','Signature','global_feature'],labels)) | dict(tested_features=g.key.nunique(),candidate_features=c.key.nunique(),tested_genes=g.gene.nunique(),candidate_genes=c.gene.nunique(),positive_candidate_genes=c[c.direction.eq('positive')].gene.nunique(),negative_candidate_genes=c[c.direction.eq('negative')].gene.nunique()))
SC=pd.DataFrame(rows)
SC['candidate_feature_fraction']=SC.candidate_features/SC.tested_features
SC['candidate_gene_fraction']=SC.candidate_genes/SC.tested_genes
export('signature_context_tested_background',SC,'Descriptive overlap, not enrichment; genes can have opposing features')
export('candidate_signature_links',L[L.candidate].drop_duplicates(),'Full mapping preserves overlapping annotations')


In [ ]:
precision=Q.groupby('assay').agg(median_n=('n_patients','median'),min_n=('n_patients','min'),max_n=('n_patients','max'),median_hdi_width=('hdi_width','median'),median_absolute_mean=('absolute_mean','median')).reset_index()
export('precision_summary',precision,'Descriptive precision, not power or calibration')
fig,axs=plt.subplots(1,2,figsize=(10,4))
for ax,(assay,g) in zip(axs,Q.groupby('assay',sort=True)):
    ax.scatter(g.n_patients,g.hdi_width,c=np.where(g.candidate,'#D81B60','#9DA3AA'),s=10,alpha=.5)
    ax.set(title=assay,xlabel='Paired patients',ylabel='Conditional slope 95% HDI width')
plot_save('precision_vs_coverage')

print('Computed candidate counts:', C.groupby('assay').size().to_dict())
print('Opposing-direction candidate genes:',G[G.opposing_candidate_directions][['assay','gene']].to_dict('records'))


In [ ]:
lines=['# PDAC PTM deeper analysis', '', 'Post hoc analysis of completed fits; no models refitted and no multiplicity-controlled discoveries declared.', '', '## Scope', mdtable(summary), '', '## Marginal versus conditional interval classifications', mdtable(trans)]
lines += ['', '## Within-protein patterns', 'Counts below are descriptive: correlated features are not independent replications and no protein-level effect was fitted.', mdtable(G[G.candidate_features>0])]
lines += ['', '## Candidate diagnostic review', f'{len(C)} candidates; {int((C.predictive_flags>0).sum())} with primary predictive flags; {int((C.unresolved_deletions>0).sum())} with unresolved deletion checks.', f'{C[C.deletion_pairs>0].shape[0]} candidates triggered deletion checks; absence of a trigger is not comprehensive deletion validation.']
if len(CD): lines += [f'Among {len(CD)} available candidate deletion comparisons, {int(CD.interval_class_changed.sum())} changed interval classification; maximum absolute posterior-mean change = {CD.absolute_mean_change.max():.4f}.']
lines += ['', '## Sensitivity coverage', f'{int(gaps.available_variants.eq("none").sum())} candidates have no additional continuous sensitivity variants in this archive.', 'Stage/purity adjustments must be assessed against their own matched-subset baselines. Newly highlighted genes are selected after inspecting results; any additional sensitivity analysis must be described as post hoc.', '', '## Interpretation', 'The data distinguish marginal modified-feature abundance associations from associations conditional on measured parent protein. Consistent directions across features and opposing directions within proteins are useful descriptive patterns. Neither proves a protein-level effect, glycan redistribution, kinase activity, or occupancy. Bulk cell composition and measurement error remain possible explanations.', 'Candidate fractions and signature overlaps do not establish a global excess of BMI associations or pathway enrichment. Formal calibration, multiplicity control and external replication are absent. Overlapping signatures are not independent confirmations.', '', '## Before redesigning the figures', 'Use the complete candidate tables and gene backgrounds, show diagnostic flags, and keep uncertain features visible. Highlight examples using a stated exploratory rule. Keep all estimates with intervals. Separate continuous, categorical and residual-correlation units.', '', '## Additional data/fits needed', '- Joint posterior draws for uncertainty in new contrasts; do not subtract HDI endpoints.', '- Patient-level data and new fits for stage/purity sensitivities of newly selected genes.', '- Localized, accession-resolved sites and external substrate annotations for kinase inference.', '- Independent data or explicitly designed calibration for broader discovery claims.', '', '## Reproducibility', 'See provenance.json, analysis_manifest.csv and the original aggregate report included in the archive. No original result files were modified.']
(OUT/'ANALYSIS_REPORT.md').write_text('\n'.join(lines),encoding='utf-8')
pd.DataFrame(MANIFEST).to_csv(OUT/'analysis_manifest.csv',index=False)
provenance=dict(input_path=str(INPUT_ZIP),input_sha256=hashlib.sha256(INPUT_ZIP.read_bytes()).hexdigest(),created_utc=STAMP,python=platform.python_version(),pandas=pd.__version__,numpy=np.__version__,completion=COMPLETION,analysis_type='post hoc descriptive audit; original model unchanged')
(OUT/'provenance.json').write_text(json.dumps(provenance,indent=2))
(OUT/'source_contract.json').write_text(json.dumps(CONTRACT,indent=2))
archive=OUT.parent/(OUT.name+'.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob('*')):
        if p.is_file(): z.write(p,p.relative_to(OUT))
    z.write(INPUT_ZIP,'source/Protein_PTM_reports.zip')
display(Markdown('\n'.join(lines[:8])))
print('Analysis folder:',OUT); print('Report ZIP:',archive)
if DOWNLOAD:
    from google.colab import files
    files.download(str(archive))
